# Real Project: Groundwater-Extraction Subsidence Monitoring with InSAR

This notebook is a **complete, runnable, real-world InSAR project**: measuring
land subsidence caused by groundwater extraction, one of the most common
operational applications of InSAR worldwide (Mexico City, Jakarta, California's
Central Valley, and Iran's Fars Province all show 10-30 cm/year of measured
subsidence from this exact technique).

**What this notebook does:**
1. Acquires a stack of Sentinel-1 SLC scenes over a subsidence-prone AOI
2. Builds a well-connected SBAS interferogram network
3. Processes every pair through the full InSAR chain (coregistration -> interferogram -> unwrap -> reference)
4. Inverts the network into a **displacement time series** and **velocity map**
5. Interprets the results against known subsidence-monitoring thresholds

**Runtime note:** This notebook generates synthetic-but-physically-realistic
SLC data reproducing a known subsidence signal, so it runs completely
end-to-end without requiring live downloads. Section 9 shows exactly how to
swap in real downloaded data.

**A critical InSAR practice demonstrated here:** phase unwrapping (SNAPHU)
recovers phase only relative to an arbitrary per-interferogram integer-cycle
offset — there is no absolute phase reference. Combining multiple
independently-unwrapped interferograms into an SBAS network **requires**
referencing every interferogram to a common stable pixel first
(Berardino et al. 2002, Section II). `SBASTimeSeries.invert()` handles
this automatically via its `reference_pixel` parameter — Section 5 shows
why this step is essential by comparing results with and without it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds

from pygeofetch.insar import (
    InterferogramGenerator,
    PhaseUnwrapper,
    AtmosphericCorrector,
    SBASTimeSeries,
)
from pygeofetch.insar.timeseries import InterferogramPair

np.random.seed(7)
WAVELENGTH_M = 0.05546576  # Sentinel-1 C-band

print("PyGeoFetch InSAR chain imported successfully")


## 1. Define the Area of Interest and Acquisition Timeline

We simulate a 6-scene Sentinel-1 stack spanning ~2 months (the standard
12-day repeat cycle), over an AOI containing a groundwater well field with
a known, realistic subsidence bowl.


In [ ]:
H, W = 120, 120   # pixels
CRS_EPSG = 4326
BOUNDS = (-99.15, 19.35, -99.05, 19.45)   # example AOI, Mexico City metro area

dates = [f"2026-{m:02d}-{d:02d}" for m, d in [
    (1,1), (1,13), (1,25), (2,6), (2,18), (3,2),
]]
n_dates = len(dates)
print(f"Acquisition dates ({n_dates} scenes, {(len(dates)-1)*12} day span):")
for d in dates:
    print(f"   {d}")


In [ ]:
def days_since(d0, d1):
    from datetime import datetime
    fmt = "%Y-%m-%d"
    return (datetime.strptime(d1, fmt) - datetime.strptime(d0, fmt)).days

# ── Build a realistic ground-truth subsidence bowl ────────────────────────
y, x = np.mgrid[0:H, 0:W]
cy, cx = H * 0.55, W * 0.45          # subsidence bowl center (near the well field)
r = np.sqrt((y - cy)**2 + (x - cx)**2)

# Gaussian-shaped subsidence bowl, peak rate -100 mm/year (realistic for
# heavily-pumped aquifers e.g. Mexico City, Jakarta). Bowl radius chosen
# wide enough that adjacent-date phase gradients stay well within SNAPHU's
# reliable unwrapping range (a standard InSAR network-design consideration).
peak_rate_mm_yr = -100.0
bowl_radius_px = 28
true_velocity_mm_yr = peak_rate_mm_yr * np.exp(-(r**2) / (2 * bowl_radius_px**2))

# A stable reference area — bedrock outcrop far from the well field,
# assumed to have zero deformation (standard InSAR practice: pick a
# reference point known to be geologically stable)
REF_PIXEL = (10, 10)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(true_velocity_mm_yr, cmap="RdBu_r", vmin=-100, vmax=20)
ax.plot(REF_PIXEL[1], REF_PIXEL[0], "k^", markersize=12, label="Reference pixel (stable)")
ax.set_title("Ground truth: synthetic subsidence velocity (mm/year)")
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, label="mm/year", fraction=0.046)
plt.tight_layout()
plt.show()

print(f"Peak subsidence rate: {true_velocity_mm_yr.min():.1f} mm/year")
print(f"(For reference: Mexico City measures up to -300mm/yr in the worst zones;")
print(f" -100mm/yr is a realistic moderate-to-severe rate)")


## 2. Simulate the SLC Stack

We generate complex SLC pairs consistent with the true subsidence signal,
including realistic speckle (Rayleigh-distributed amplitude) and phase
decorrelation noise — this is what makes the synthetic data behave like
real Sentinel-1 acquisitions rather than an idealized noise-free test case.


In [ ]:
def make_slc(true_disp_mm, decorrelation=0.05):
    """
    Generate a synthetic complex SLC consistent with a given cumulative
    displacement field (mm), with realistic speckle and decorrelation.
    """
    amp = np.random.rayleigh(scale=80, size=(H, W)).astype(np.float32)
    disp_m = true_disp_mm / 1000.0
    phase = -4 * np.pi / WAVELENGTH_M * disp_m
    noise = np.random.normal(0, decorrelation * np.pi, size=(H, W))
    slc = (amp * np.exp(1j * (phase + noise))).astype(np.complex64)
    return slc

# Cumulative displacement at each date (linear velocity, no seasonal term
# for this network size — keeps validation numbers easy to interpret)
cumulative_disp = {}
for d in dates:
    t_years = days_since(dates[0], d) / 365.25
    cumulative_disp[d] = true_velocity_mm_yr * t_years

slc_stack = {d: make_slc(cumulative_disp[d]) for d in dates}
print(f"Generated {len(slc_stack)} synthetic SLC acquisitions")
print(f"Cumulative displacement at final date: "
      f"[{cumulative_disp[dates[-1]].min():.1f}, {cumulative_disp[dates[-1]].max():.1f}] mm")


## 3. Build the SBAS Interferogram Network

A **well-connected network** is essential — every date should be reachable
through short-baseline pairs. The standard strategy: connect each
acquisition to its next 1-2 neighbours in time, plus a few longer
skip-connections for redundancy (this matches MintPy/ISCE's default
network-generation strategy).


In [ ]:
pair_indices = []
for i in range(n_dates - 1):
    pair_indices.append((i, i + 1))          # sequential (12-day baseline)
    if i + 2 < n_dates:
        pair_indices.append((i, i + 2))      # skip-1 (24-day baseline, redundancy)

print(f"SBAS network: {len(pair_indices)} interferometric pairs, {n_dates} dates")

# Connectivity check via union-find
parent = list(range(n_dates))
def find(x):
    while parent[x] != x:
        x = parent[x]
    return x
def union(a, b):
    parent[find(a)] = find(b)
for i, j in pair_indices:
    union(i, j)
print(f"Every date reachable from every other: {len(set(find(i) for i in range(n_dates))) == 1}")


## 4. Process Every Pair Through the InSAR Chain

For each pair: form the interferogram, estimate coherence, and unwrap the
phase. We track per-pair quality (coherence and unwrapping reliability)
and skip any pair that doesn't meet quality thresholds — standard InSAR
network quality control.


In [ ]:
gen        = InterferogramGenerator(coherence_window=5, esd_enabled=False)  # ESD n/a for this synthetic, already-deburst data
unwrapper  = PhaseUnwrapper(cost_mode="defo", init_method="mcf")

sbas_pairs = []
skipped = []

for k, (i, j) in enumerate(pair_indices):
    date_ref, date_sec = dates[i], dates[j]
    ref_slc, sec_slc = slc_stack[date_ref], slc_stack[date_sec]

    # Form interferogram + estimate coherence
    interferogram = ref_slc * np.conj(sec_slc)
    coherence = gen._estimate_coherence(ref_slc, sec_slc, window=5)

    # Unwrap
    try:
        unwrapped, conncomp = unwrapper.unwrap(interferogram, coherence, nlooks=4.0)
    except ImportError:
        print("snaphu-py not installed — install with: pip install \"pygeofetch[insar]\"")
        break

    reliable_pct = 100 * np.mean(conncomp > 0)
    mean_coh = float(coherence.mean())

    if mean_coh < 0.3 or reliable_pct < 50:
        skipped.append((date_ref, date_sec, mean_coh, reliable_pct))
        continue

    sbas_pairs.append(InterferogramPair(
        reference_date=date_ref,
        secondary_date=date_sec,
        unwrapped_phase=unwrapped,
        coherence=coherence,
    ))

    print(f"  [{k+1}/{len(pair_indices)}] {date_ref} -> {date_sec}  "
          f"coherence={mean_coh:.3f}  reliable={reliable_pct:.0f}%")

print(f"\nProcessed {len(sbas_pairs)}/{len(pair_indices)} pairs successfully")
if skipped:
    print(f"Skipped {len(skipped)} low-quality pairs")


## 5. Why Reference-Pixel Normalization Matters

Before running the SBAS inversion, let's demonstrate **why** the
`reference_pixel` parameter is essential — this is a genuine, common
InSAR pitfall, not a hypothetical concern.

Each interferogram's unwrapped phase is only known up to an arbitrary
constant (SNAPHU has no way to know the *absolute* phase — only relative
phase differences within one interferogram are physically meaningful).
When you combine multiple **independently unwrapped** interferograms
into a joint least-squares inversion without first anchoring them all to
the same reference point, these arbitrary per-interferogram offsets
corrupt the solution.


In [ ]:
sbas = SBASTimeSeries(wavelength_m=WAVELENGTH_M, reference_date=dates[0])

# A BADLY CHOSEN reference point: inside the deforming bowl itself.
# In real projects this happens when analysts pick a "convenient" pixel
# (e.g. image center) without checking whether it's actually stable.
bad_reference_pixel = (int(cy), int(cx))  # the bowl center itself!

result_bad_ref = sbas.invert(sbas_pairs, coherence_threshold=0.25,
                              reference_pixel=bad_reference_pixel)

# A verified-stable reference pixel, far outside the deformation zone
result_good_ref = sbas.invert(sbas_pairs, coherence_threshold=0.25,
                               reference_pixel=REF_PIXEL)

vel_bad_mm  = result_bad_ref.velocity * 1000
vel_good_mm = result_good_ref.velocity * 1000

rmse_bad  = np.sqrt(np.mean((vel_bad_mm  - true_velocity_mm_yr)**2))
rmse_good = np.sqrt(np.mean((vel_good_mm - true_velocity_mm_yr)**2))

print(f"RMSE with reference_pixel={bad_reference_pixel} [INSIDE the bowl — bad choice]: {rmse_bad:.2f} mm/year")
print(f"RMSE with reference_pixel={REF_PIXEL} [verified stable]:                     {rmse_good:.2f} mm/year")
print()
print(f"Bowl peak with bad reference:  {vel_bad_mm.min():.1f} mm/yr  (true: {true_velocity_mm_yr.min():.1f})")
print(f"Bowl peak with good reference: {vel_good_mm.min():.1f} mm/yr  (true: {true_velocity_mm_yr.min():.1f})")
print()
print("Takeaway: referencing to a pixel that ITSELF is subsiding shifts the")
print("apparent zero-point into the deforming area, systematically biasing")
print("every other pixel's estimated velocity by the reference pixel's own")
print("(non-zero) true rate. ALWAYS verify your reference pixel is truly")
print("stable — bedrock, a monitored benchmark, or a point confirmed to be")
print("far outside any expected deformation.")


## 6. Invert the Network into a Displacement Time Series

This is the core SBAS step (Berardino et al. 2002): a weighted
least-squares inversion that converts the network of pairwise phase
differences into a per-date, per-pixel displacement time series relative
to a reference date and a stable reference pixel.


In [ ]:
ts_result = sbas.invert(sbas_pairs, coherence_threshold=0.25, reference_pixel=REF_PIXEL)

print(f"Time series inverted:")
print(f"  Dates: {len(ts_result.dates)}")
print(f"  Displacement stack shape: {ts_result.displacement.shape}")
print(f"  Reference date: {ts_result.reference_date}")
print(f"  Reference pixel value (should be exactly 0): "
      f"{ts_result.velocity[REF_PIXEL]*1000:.4f} mm/year")
print(f"  Mean inversion residual RMS: {ts_result.residual_rms.mean():.6f} m")


## 7. Compare Recovered Velocity Against Ground Truth

Since we know the true subsidence signal we injected, we can directly
validate the SBAS inversion's accuracy — this is exactly the kind of
verification you should do before trusting a real InSAR result.


In [ ]:
recovered_velocity_mm_yr = ts_result.velocity * 1000

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

im0 = axes[0].imshow(true_velocity_mm_yr, cmap="RdBu_r", vmin=-100, vmax=20)
axes[0].plot(REF_PIXEL[1], REF_PIXEL[0], "k^", markersize=10)
axes[0].set_title("Ground truth velocity")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="mm/year")

im1 = axes[1].imshow(recovered_velocity_mm_yr, cmap="RdBu_r", vmin=-100, vmax=20)
axes[1].plot(REF_PIXEL[1], REF_PIXEL[0], "k^", markersize=10)
axes[1].set_title("SBAS-recovered velocity")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="mm/year")

error = recovered_velocity_mm_yr - true_velocity_mm_yr
im2 = axes[2].imshow(error, cmap="RdBu_r", vmin=-15, vmax=15)
axes[2].set_title("Error (recovered - truth)")
plt.colorbar(im2, ax=axes[2], fraction=0.046, label="mm/year")

plt.tight_layout()
plt.savefig("sbas_validation.png", dpi=120, bbox_inches="tight")
plt.show()

rmse = np.sqrt(np.mean(error**2))
peak_error = np.abs(error).max()
print(f"\nValidation metrics:")
print(f"  RMSE:            {rmse:.2f} mm/year")
print(f"  Peak abs. error: {peak_error:.2f} mm/year")
print(f"  Bowl peak — true: {true_velocity_mm_yr.min():.1f} mm/yr, "
      f"recovered: {recovered_velocity_mm_yr.min():.1f} mm/yr")
print()
print(f"  (Residual error reflects the realistic decorrelation noise injected")
print(f"   in Section 2 — real Sentinel-1 InSAR shows comparable per-pixel")
print(f"   scatter, which is why operational products are typically spatially")
print(f"   filtered/multi-looked before interpretation, as done in Section 8.)")


## 8. Time Series at the Point of Maximum Subsidence

A key deliverable for stakeholders (water authorities, infrastructure
engineers) is the displacement time series at specific points of interest
— typically monitoring wells, buildings, or infrastructure corridors.


In [ ]:
# Find pixel with maximum subsidence
peak_y, peak_x = np.unravel_index(np.argmin(true_velocity_mm_yr), true_velocity_mm_yr.shape)

ts_at_peak_mm = ts_result.displacement[:, peak_y, peak_x] * 1000
true_ts_at_peak_mm = [cumulative_disp[d][peak_y, peak_x] for d in dates]

t_years_axis = [days_since(dates[0], d) / 365.25 for d in ts_result.dates]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_years_axis, true_ts_at_peak_mm, 'o-', label="Ground truth", color="black", linewidth=2)
ax.plot(t_years_axis, ts_at_peak_mm, 's--', label="SBAS recovered", color="crimson", markersize=6)
ax.set_xlabel("Time (years since first acquisition)")
ax.set_ylabel("Cumulative LOS displacement (mm)")
ax.set_title(f"Displacement time series at subsidence bowl center (pixel {peak_y},{peak_x})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("timeseries_at_peak.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nFinal cumulative displacement at monitoring point:")
print(f"  Ground truth: {true_ts_at_peak_mm[-1]:.1f} mm")
print(f"  SBAS recovered: {ts_at_peak_mm[-1]:.1f} mm")


## 9. Spatial Filtering for Interpretation

Individual pixels carry realistic decorrelation noise (as they do in real
InSAR products). A standard operational step — applied by every major
InSAR processing chain from GAMMA to MintPy — is spatial averaging
(multilooking / boxcar filtering) before final interpretation and
classification, since deformation signals of interest are almost always
spatially coherent over tens to hundreds of metres, while decorrelation
noise is not.


In [ ]:
from scipy.ndimage import uniform_filter

smoothed_velocity_mm_yr = uniform_filter(recovered_velocity_mm_yr, size=7)

rmse_smoothed = np.sqrt(np.mean((smoothed_velocity_mm_yr - true_velocity_mm_yr)**2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
im0 = axes[0].imshow(recovered_velocity_mm_yr, cmap="RdBu_r", vmin=-100, vmax=20)
axes[0].set_title(f"Raw (RMSE={rmse:.1f} mm/yr)")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="mm/year")

im1 = axes[1].imshow(smoothed_velocity_mm_yr, cmap="RdBu_r", vmin=-100, vmax=20)
axes[1].set_title(f"7x7 spatially filtered (RMSE={rmse_smoothed:.1f} mm/yr)")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="mm/year")

plt.tight_layout()
plt.show()

print(f"RMSE improvement from spatial filtering: {rmse:.2f} -> {rmse_smoothed:.2f} mm/year")


## 10. Interpreting the Results — Operational Thresholds

Real subsidence-monitoring programs use velocity thresholds to trigger
different responses. Typical thresholds used by groundwater authorities
(adapted from published Mexico City, Jakarta, and California Central
Valley monitoring programs):

| Velocity | Classification | Typical Response |
|---|---|---|
| > -5 mm/yr | Stable | Routine monitoring |
| -5 to -20 mm/yr | Slow subsidence | Increased monitoring frequency |
| -20 to -50 mm/yr | Moderate | Infrastructure inspection triggered |
| -50 to -100 mm/yr | Severe | Groundwater extraction limits considered |
| < -100 mm/yr | Critical | Emergency infrastructure assessment |


In [ ]:
thresholds = [
    (-5, "Stable"),
    (-20, "Slow subsidence"),
    (-50, "Moderate"),
    (-100, "Severe"),
    (-np.inf, "Critical"),
]

def classify(v):
    for thresh, label in thresholds:
        if v >= thresh:
            return label
    return "Critical"

classification = np.vectorize(classify)(smoothed_velocity_mm_yr)

unique, counts = np.unique(classification, return_counts=True)
total_px = classification.size

print("Subsidence classification breakdown (% of AOI, spatially filtered map):\n")
for label in ["Stable", "Slow subsidence", "Moderate", "Severe", "Critical"]:
    if label in unique:
        pct = 100 * counts[list(unique).index(label)] / total_px
        print(f"  {label:20s}  {pct:5.1f}%")

print(f"\n  Peak measured rate: {smoothed_velocity_mm_yr.min():.1f} mm/year")
print(f"  -> Classification: {classify(smoothed_velocity_mm_yr.min())}")
if smoothed_velocity_mm_yr.min() < -50:
    print(f"  -> RECOMMENDATION: Groundwater extraction limits should be")
    print(f"     considered in the bowl center. Cross-reference with")
    print(f"     extraction well locations and building/pipeline positions.")


In [ ]:
# Save all deliverables
output_dir = Path("./subsidence_monitoring_output")
saved = ts_result.save(output_dir, profile={"crs": CRS.from_epsg(CRS_EPSG),
                                             "transform": from_bounds(*BOUNDS, W, H)})
for name, path in saved.items():
    print(f"  Saved: {path}")

print(f"\nDeliverables ready for GIS integration:")
print(f"  - velocity_m_per_year.tif       : velocity map for classification/mapping")
print(f"  - displacement_timeseries.tif   : full time series, one band per date")
print(f"  - residual_rms.tif              : inversion quality / confidence map")


## 11. Using Real Sentinel-1 Data Instead

This notebook used synthetic data so it runs end-to-end without downloads.
To run this exact pipeline on **real data**, replace Sections 1-2 with:

```python
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions

client = PyGeoFetch()
query = SearchQuery(
    bbox=BoundingBox.from_string("-99.15,19.35,-99.05,19.45"),
    start_date="2026-01-01", end_date="2026-03-15",
    product_type="SLC", polarisation="VV",
    pass_direction="DESCENDING",   # keep orbit direction consistent!
    max_results=10,
)
results = client.search(query, providers=["copernicus"])
client.download(results, destination=Path("./data/"),
                options=DownloadOptions(parallel=4))

# Fetch precise orbit files for every scene
for scene in results:
    client.fetch_orbit_file(scene.id)
```

Then proceed with Sections 3-10 exactly as written — the only changes are:
1. `slc_stack[date]` points to the actual extracted, complex-valued
   measurement TIFFs from each downloaded `.SAFE` product instead of the
   synthetic arrays from Section 2.
2. **Choosing `REF_PIXEL`**: for real data, pick a pixel corresponding to
   a genuinely stable feature visible in your AOI — bedrock outcrop, a
   large building rooftop, or a surveyed geodetic benchmark. Verify its
   stability independently (e.g. against GPS/leveling data) whenever
   possible; Section 5 showed how much a poorly-chosen reference point can
   bias the entire result.

**Important real-data considerations:**
- Keep `pass_direction` consistent (all ascending or all descending) —
  mixing orbit geometries in one SBAS network is a common and serious error.
- Extract the correct sub-swath/burst covering your AOI before processing —
  Sentinel-1 IW SLC products span 3 sub-swaths, each with many bursts.
- Real coherence will typically be lower than the synthetic values here,
  especially over vegetation — expect to filter more pairs by coherence
  threshold.
- A DEM is needed for real topographic phase removal — supply one via
  `InterferogramGenerator.process_pair(..., dem=dem_path)`. This notebook's
  synthetic SLC generator has no genuine elevation-correlated phase
  component, so the DEM step was omitted here for clarity; real Sentinel-1
  interferograms always require it.
